# Stage 13 - Productization

In [7]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install scikit-learn
# !pip install joblib
# !pip install flask
# !pip install requests

## 1. Generate data and train a model

Nothing to fill in here — run it. Note `os.makedirs` **before** `joblib.dump`: without it
the save fails with `FileNotFoundError`, because `model/` does not exist yet.

In [8]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# the dataset for this homework - generated, not loaded
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

model = LinearRegression()
model.fit(X, y)

os.makedirs('model', exist_ok=True)          # BEFORE the dump, not after
joblib.dump(model, 'model/model.pkl')

# prove the file on disk is usable: load it back and predict with the loaded copy
reloaded = joblib.load('model/model.pkl')
print('saved to model/model.pkl')
print('prediction from the reloaded model:', reloaded.predict([[0.1, 0.2]])[0])

saved to model/model.pkl
prediction from the reloaded model: 23.58961171297328


## 2. Write `app.py`

Run the cell — it writes `app.py` to disk. Both routes validate their input and return a
JSON `{"error": ...}` with HTTP 400 (never a traceback) when `features` is missing / the
wrong length / not numeric, or a path parameter is not a number.

**The model load stays where it is**, at the top of the file. It runs once when the app
starts. Do not move it inside a route: a route that loads the model on every request
re-reads the file from disk for every single caller.

In [9]:
app_code = '''from flask import Flask, request, jsonify
import joblib

# loaded ONCE, at startup - not inside a route
model = joblib.load('model/model.pkl')
app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}
    features = data.get('features')
    if not isinstance(features, list) or len(features) != 2:
        return jsonify({'error': 'send JSON {"features": [f1, f2]} with exactly 2 numbers'}), 400
    try:
        row = [float(x) for x in features]
    except (TypeError, ValueError):
        return jsonify({'error': 'both features must be numbers'}), 400
    prediction = model.predict([row])[0]
    return jsonify({'prediction': float(prediction)})


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):
    try:
        row = [float(f1), float(f2)]
    except ValueError:
        return jsonify({'error': 'both path values must be numbers'}), 400
    prediction = model.predict([row])[0]
    return jsonify({'prediction': float(prediction)})


if __name__ == '__main__':
    # port 5001, not 5000: on macOS the Control Center / AirPlay Receiver also
    # listens on 5000 and answers first with HTTP 403.
    app.run(port=5001)
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('wrote app.py')

wrote app.py


## 3. Launch the server

Starts Flask (`app.py`) as a background process on `http://127.0.0.1:5001`. Leave it
running. Every time you change `app.py`, run `flask_proc.terminate()` and then this cell
again.

In [10]:
import subprocess, sys, time

flask_proc = subprocess.Popen([sys.executable, 'app.py'])
time.sleep(2)                       # give Werkzeug a moment to bind the port
print('Flask started (pid', flask_proc.pid, ') on http://127.0.0.1:5001')

 * Serving Flask app 'app'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Flask started (pid 39182 ) on http://127.0.0.1:5001


## 4. Call your own API

The POST route, the GET route, and two deliberately bad calls (bad path parameter, and a
POST body with no `features` key). **Leave this output visible in the notebook you submit —
it is your testing evidence.**

In [11]:
import requests

BASE = 'http://127.0.0.1:5001'

try:
    r1 = requests.post(BASE + '/predict', json={'features': [0.1, 0.2]}, timeout=5)
    print('POST /predict            ', r1.status_code, r1.text.strip())

    r2 = requests.get(BASE + '/predict/0.1/0.2', timeout=5)
    print('GET  /predict/0.1/0.2    ', r2.status_code, r2.text.strip())

    # deliberately bad: not a number. This must be a 400 and a JSON error,
    # not a traceback in the server window.
    r3 = requests.get(BASE + '/predict/abc/0.2', timeout=5)
    print('GET  /predict/abc/0.2    ', r3.status_code, r3.text.strip())

    # deliberately bad: POST body missing the "features" key -> also 400
    r4 = requests.post(BASE + '/predict', json={}, timeout=5)
    print('POST /predict {}         ', r4.status_code, r4.text.strip())
except requests.exceptions.ConnectionError:
    print('No server on port 5001. Run the launch cell above, wait a few seconds,')
    print('then run this cell again.')

POST /predict             200 {"prediction":23.58961171297328}
GET  /predict/0.1/0.2     200 {"prediction":23.58961171297328}
GET  /predict/abc/0.2     400 {"error":"both path values must be numbers"}
POST /predict {}          400 {"error":"send JSON {\"features\": [f1, f2]} with exactly 2 numbers"}


127.0.0.1 - - [30/Aug/2026 17:51:23] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [30/Aug/2026 17:51:23] "GET /predict/0.1/0.2 HTTP/1.1" 200 -
127.0.0.1 - - [30/Aug/2026 17:51:23] "GET /predict/abc/0.2 HTTP/1.1" 400 -
127.0.0.1 - - [30/Aug/2026 17:51:23] "POST /predict HTTP/1.1" 400 -


## 5. Write `README.md`

Run the cell — it writes `README.md` with the model description, the start command, a
copy-pasteable `curl` for each route with the response it produced above, and the
bad-input behavior.

In [12]:
readme = '''# Stage 13 Homework - Prediction API

A `LinearRegression` model trained on a synthetic 2-feature dataset from scikit-learn's
`make_regression` (100 samples, noise=0.1, random_state=42). It takes two numeric features
and returns a single continuous predicted value.

## Running it

    python app.py

The server starts on http://127.0.0.1:5001 and loads model/model.pkl once at startup.
(Port 5001, not 5000: on macOS the Control Center / AirPlay Receiver also listens on 5000.)

## POST /predict

    curl -X POST http://127.0.0.1:5001/predict \\
         -H "Content-Type: application/json" \\
         -d "{\\"features\\": [0.1, 0.2]}"

Response: 200 {"prediction":23.58961171297328}

## GET /predict/<f1>/<f2>

    curl http://127.0.0.1:5001/predict/0.1/0.2

Response: 200 {"prediction":23.58961171297328}

## Bad input

Every bad request returns HTTP 400 with a JSON `error` field instead of a traceback:

- `GET /predict/abc/0.2` (not a number) -> {"error":"both path values must be numbers"}
- `POST /predict` with `features` missing or not exactly 2 values ->
  {"error":"send JSON {\\"features\\": [f1, f2]} with exactly 2 numbers"}
- `POST /predict` with non-numeric values -> {"error":"both features must be numbers"}
'''

with open('README.md', 'w') as f:
    f.write(readme)
print('wrote README.md')

wrote README.md
